# Dam removal effects by distance downstream

**Question:** after a dam is removed, does water quality change downstream, how far does the change reach, and is it bigger for bigger reservoirs?

**How to read every result**
- **Effect** = change after removal, in the units shown (log10 units for skewed measures: 0.1 = about +26%).
- **95% range** = plausible values. If it includes 0, we can't tell whether there was a change.
- **Verdict:** *detected* (range excludes 0) · *no meaningful change* (whole range is inside the "too small to matter" threshold) · *inconclusive* (can't tell).
- **BACI** (downstream change minus the upstream change at the same dam) is the strong evidence. **Before/after** (downstream only) is supporting evidence: it can't separate the removal from regional trends.
- Results are associations, not proof of cause. Fewer than 15 dams = *exploratory*.

Run all cells; nothing is refit (the cluster jobs `fit_distance_models.slurm` and `fit_stage4.slurm` produce the inputs).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Find the project folder and load the report tables written by the cluster jobs.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
REPORT_DIR, INFLUENCE_DIR = MODEL_DIR / "stage4" / "report", MODEL_DIR / "stage4" / "influence"
if not (REPORT_DIR / "model_summary.csv").exists():      # rebuild tables if missing (seconds)
    rep.build(MODEL_DIR, REPORT_DIR, figures=False)

def read(name):
    path = REPORT_DIR / f"{name}.csv"
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 1 else pd.DataFrame()

summary, by_distance, curve = read("model_summary"), read("effects_by_distance"), read("effects_curve")
sensitivity, comparison, influence = read("sensitivity_table"), read("distance_comparison"), read("influence_summary")
thresholds, threshold_sensitivity = read("sesoi_thresholds"), read("sesoi_sensitivity")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.3g}")

# Plain column names for the tables below.
PLAIN = {"effect_at_reference": "effect at 5 km", "effect_ci_low": "low", "effect_ci_high": "high",
         "mde80_effect": "smallest detectable", "effect_class": "verdict",
         "modifier_slope": "change with distance", "modifier_ci_low": "slope low",
         "modifier_ci_high": "slope high", "slope_class": "distance verdict", "sample_flag": "note"}
# Discharge rows are a sampling check (section 9), kept out of the main tables.
is_check = (summary["purpose"].eq("sampling-conditions check") if "purpose" in summary
            else pd.Series(False, index=summary.index))
primary = summary[(summary["run"] == rep.PRIMARY[0]) & (summary["re"] == rep.PRIMARY[1]) & ~is_check]
check = summary[(summary["run"] == rep.PRIMARY[0]) & (summary["re"] == rep.PRIMARY[1]) & is_check]
print(f"Model outputs: {MODEL_DIR}")

## 1. Main results

One row per water-quality measure. **Effect at 5 km** with its 95% range and verdict; **change with distance** says whether the effect shrinks or grows going downstream (negative = a positive effect fades). **Smallest detectable** = how big a real change would have to be for this model to reliably see it: if the effect is inconclusive and this number is large, the data are too thin, not evidence of no effect.

In [ ]:
cols = ["series", "dams", "effect_at_reference", "effect_ci_low", "effect_ci_high", "mde80_effect",
        "effect_class", "modifier_slope", "modifier_ci_low", "modifier_ci_high", "slope_class", "sample_flag"]
for spec, label in [("baci_distance", "BACI (strong evidence)"), ("before_after_distance", "Before/after (supporting)")]:
    print(label)
    fitted = primary[(primary["spec"] == spec) & primary["error"].fillna("").eq("")]
    display(fitted[cols].fillna({"sample_flag": ""}).rename(columns=PLAIN))
    skipped = primary[(primary["spec"] == spec) & primary["error"].fillna("").ne("")]
    if len(skipped):
        print("  not fit (too few dams):", ", ".join(skipped["series"]))

## 2. How the effect changes with distance

**Line** = best estimate; **shaded band** = 95% range; **grey band** = too small to matter. Where the shaded band covers 0 we can't tell; a line moving toward 0 going right means the effect fades downstream. Dashed = simpler model (sensitivity). Left: BACI; right: before/after.

In [ ]:
for series in dict.fromkeys(curve["series"]):
    if series in rep.DIAGNOSTIC_SERIES:          # shown in section 9
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    rep.plot_distance_response(curve, by_distance, series, axes, rep.SESOI.get(series))
    fig.tight_layout(); plt.show()

**Effects at 1, 5, 10 and 20 km** (percent columns for skewed measures). Distances where no stations were measured are left out.

In [ ]:
pts = by_distance[by_distance["primary"] & by_distance["distance_km"].notna()
                 & by_distance["spec"].isin(["baci_distance", "before_after_distance"])
                 & ~by_distance["series"].isin(rep.DIAGNOSTIC_SERIES)]
cols = ["series", "design", "distance_km", "estimate", "ci_low", "ci_high", "effect_class",
        "pct_estimate", "pct_ci_low", "pct_ci_high", "dams"]
display(pts[[c for c in cols if c in pts]].sort_values(["series", "design", "distance_km"])
        .rename(columns={"ci_low": "low", "ci_high": "high", "effect_class": "verdict",
                         "pct_estimate": "% change", "pct_ci_low": "% low", "pct_ci_high": "% high"}))

## 3. Do the results depend on our choices?

Each column re-runs the model a different way: the simpler random-effects model, only stations where the dam supplies most of the water (ratio limits 100 / 10 / 2), and with the 12 suspect dams kept in. **Similar numbers across a row = a robust result.** `n` = dams.

In [ ]:
for km in (1, 5, 20):
    t = sensitivity[sensitivity["distance_km"] == km].copy()
    if t.empty:
        continue
    t["cell"] = (t["estimate"].map("{:+.3f}".format) + " [" + t["ci_low"].map("{:+.3f}".format) + ", "
                 + t["ci_high"].map("{:+.3f}".format) + "] n=" + t["dams"].astype(int).astype(str))
    print(f"Effect at {km} km")
    display(t.pivot_table(index=["series", "design"], columns="model_run", values="cell", aggfunc="first"))

## 4. One effect for every dam, or does each dam differ?

The main model lets each dam have its own effect when the data support it (`re_choice`). Where dams differ, assuming one shared effect makes the range too narrow: **width ratio > 1** shows how much more honest uncertainty the main model has. Temperature's BACI fit found no dam-to-dam difference, so it uses the simpler model.

In [ ]:
auto = summary[(summary["run"] == "primary") & (summary["re"] == "auto") & ~is_check]
width = lambda f: f["effect_ci_high"] - f["effect_ci_low"]
table = auto.copy()
for other in ("intercepts", "slopes"):
    alt = summary[(summary["run"] == "primary") & (summary["re"] == other)][["series", "spec", "effect_ci_low", "effect_ci_high"]]
    if len(alt):
        table = table.merge(alt.assign(**{f"w_{other}": width(alt)})[["series", "spec", f"w_{other}"]],
                            on=["series", "spec"], how="left")
        table[f"width ratio vs {other}"] = width(table) / table[f"w_{other}"]
cols = ["series", "spec", "dams", "re_choice", "lrt_p_method", "width ratio vs intercepts", "width ratio vs slopes"]
display(table[[c for c in cols if c in table]])

## 5. Is any result driven by a single dam?

Each bar = the change in the 5 km effect when that dam is left out, in standard errors. **Bars beyond the dotted ±1 lines (red) are influential dams.** In the table, if the bootstrap range (resampling whole dams) is close to the model range, the uncertainty is trustworthy; if they differ, trust the wider one.

In [ ]:
if len(influence):
    cols = ["series", "dams", "full_effect_at_reference", "lodo_effect_min", "lodo_effect_max",
            "influential_dams", "influential_dam_ids", "full_effect_ci_low", "full_effect_ci_high",
            "effect_at_reference_boot_low", "effect_at_reference_boot_high"]
    display(influence[[c for c in cols if c in influence]].rename(columns={
        "full_effect_at_reference": "effect at 5 km", "lodo_effect_min": "min leaving one out",
        "lodo_effect_max": "max leaving one out", "full_effect_ci_low": "model low",
        "full_effect_ci_high": "model high", "effect_at_reference_boot_low": "bootstrap low",
        "effect_at_reference_boot_high": "bootstrap high"}))
lodo_path = INFLUENCE_DIR / "lodo.csv"
if lodo_path.exists():
    lodo = pd.read_csv(lodo_path)
    names = list(dict.fromkeys(lodo["series"]))
    fig, axes = plt.subplots(len(names), 1, figsize=(12, 2 * len(names)), squeeze=False)
    for ax, s in zip(axes[:, 0], names):
        g = lodo[lodo["series"] == s].sort_values("delta_effect_at_reference_in_se")
        ax.bar(g["dam_left_out"].astype(str), g["delta_effect_at_reference_in_se"],
               color=np.where(g["influential"].fillna(False).astype(bool), "C3", "C0"))
        for y in (-1, 1):
            ax.axhline(y, ls=":", color="grey")
        ax.set_title(s, fontsize=9); ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    fig.tight_layout(); plt.show()

## 6. Distance in km, or in reservoir lengths?

Same dams and data, two ways of measuring distance. **Negative ΔAIC = reservoir lengths fit better** (bigger reservoirs reach further, as the advisor suggested); positive = km fits better; within ±2 = no real difference.

In [ ]:
cols = ["series", "design", "dams", "delta_aic_scaled_minus_raw", "preferred", "same_sample", "same_covariates"]
display(comparison[[c for c in cols if c in comparison]].rename(columns={"delta_aic_scaled_minus_raw": "ΔAIC"})
        if len(comparison) else pd.DataFrame({"note": ["no comparison run yet"]}))

## 7. Which side moved? (pH, with temperature and oxygen for comparison)

BACI effect = downstream change − upstream change, so a positive BACI result can come from downstream rising **or** upstream falling. **Left:** average values over time relative to before removal (orange = upstream controls, blue = downstream). **Right:** one dot per dam; above the dashed line = positive BACI. For pH, check whether the blue line or the orange line moved.

In [ ]:
comp_path = INFLUENCE_DIR / "components_summary.csv"
if comp_path.exists():
    display(pd.read_csv(comp_path)[["series", "quantity", "n_dams", "mean", "ci_low", "ci_high"]])
for series in ["ph", "water_temp", "dissolved_oxygen"]:
    tp, cp = INFLUENCE_DIR / f"trajectory_{series}.csv", INFLUENCE_DIR / f"components_{series}.csv"
    if not tp.exists():
        continue
    traj, comp = pd.read_csv(tp), pd.read_csv(cp)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for role, color, label in (("control", "C1", "upstream controls"), ("treated", "C0", "downstream")):
        g = traj[traj["role"] == role].sort_values("years_from_removal")
        axes[0].errorbar(g["years_from_removal"], g["mean"], yerr=1.96 * g["se"], color=color,
                         marker="o", capsize=2, label=label)
    axes[0].axvline(0, color="black", lw=0.8); axes[0].axhline(0, color="grey", lw=0.6)
    axes[0].set(xlabel="years from removal", ylabel="change vs before", title=series); axes[0].legend()
    axes[1].scatter(comp["change_upstream"], comp["change_downstream"])
    lim = np.nanmax(np.abs(comp[["change_upstream", "change_downstream"]].to_numpy())) * 1.1
    axes[1].plot([-lim, lim], [-lim, lim], color="grey", ls="--", lw=0.8)
    axes[1].set(xlabel="upstream change", ylabel="downstream change", title=f"{series}: one dot per dam")
    fig.tight_layout(); plt.show()

## 8. Bigger reservoirs, bigger effects? (exploratory)

**Size slope** = how much the effect changes per step up in reservoir size, with river size held fixed. Positive = bigger effect for bigger reservoirs. Few dams have the data, so expect wide ranges.

In [ ]:
size = summary[summary["run"].isin(["size_proxy", "size_polygon"]) & summary["re"].eq("auto") & ~is_check]
cols = ["run", "series", "spec", "dams", "modifier_slope", "modifier_ci_low", "modifier_ci_high", "slope_class", "sample_flag"]
display(size[cols].rename(columns={"modifier_slope": "size slope", "modifier_ci_low": "low",
                                   "modifier_ci_high": "high", "slope_class": "verdict", "sample_flag": "note",
                                   "run": "size measure"}))

## 9. Check: were samples taken at similar flows before and after?

Discharge (stream flow measured with each sample) is modeled like the others, but as a **check, not a finding**: small dam removals rarely change flow. **Inconclusive or near 0 = good** (before and after samples were taken under similar flows). **Detected = caution:** results for flow-sensitive measures (turbidity, sediment, conductance, oxygen) may partly reflect different flow conditions.

In [ ]:
cols = ["spec", "dams", "effect_at_reference", "effect_ci_low", "effect_ci_high", "effect_class", "error"]
display(check[[c for c in cols if c in check]].rename(columns=PLAIN) if len(check)
        else pd.DataFrame({"note": ["no discharge fits (rerun fit_distance_models.slurm)"]}))

## 10. Thresholds: what counts as "too small to matter"

These thresholds are **our own choices, not environmental standards**. The second table shows how many verdicts would change if the thresholds were half or double: those verdicts depend on the choice, so quote the numbers, not just the label.

In [ ]:
display(thresholds[["series", "threshold"]] if len(thresholds) else thresholds)
if len(threshold_sensitivity):
    changed = threshold_sensitivity[threshold_sensitivity["class_changes"]]
    print(f"{len(changed)} of {len(threshold_sensitivity)} verdicts change between 0.5x and 2x the thresholds")
    display(changed[[c for c in changed if c.startswith("class_at") or c in ("series", "design", "at")]])

**Notes:** distance is measured along the river. The drainage-area ratio (how much of the river's water comes through the dam) is an estimate, not a flow measurement. The reservoir-size measure is an index from dam height, slope and river size, not a measured volume or sediment amount. Twelve dams suspected of being placed on the wrong stream are excluded (`config/suspect_dam_matches.csv`). Full tables: `stage4/report/`.